# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# ito4.jp に統一する(2026-09-17)

**同じホスト(163.43.218.158)のまま、名前をすべて `ito4.jp` にする。** 旧 `ito8795.com` は**切り替えと同時に手放す。**

| 決めたこと | 中身 |
|---|---|
| 公開ページ | `https://ito4.jp/` |
| 管理画面 | **`https://admin.ito4.jp/`**(別オリジン。[12](12-hardening-2026-09-15.ipynb) §5 の考え方をここで当てる)。phpMyAdmin・Logto Console も `admin.ito4.jp:8281`・`:3002` |
| 旧 `ito8795.com` | **切り替えた時点で使えなくなる。** 証明書にも入れない。配布済みの APK・ブックマーク・HSTS を覚えたブラウザは繋がらなくなる |
| audience(API リソース) | **`https://ito8795.com/api` → `https://ito4.jp/api`。** Logto の識別子は変えられないので、新しいリソースを作ってスコープとロールを写す(`logto-api-resource.php`) |
| Android | **新しい APK が要る。** 接続先 `ito4.jp`・apiResource `https://ito4.jp/api`(手元は済み。両フレーバーでテスト 567 件ずつ通過)。古い APK は切り替えた時点で止まる |
| メール | 差出人 `noreply@ito4.jp`。**DKIM は旧の鍵を複製**(DNS の `mail._domainkey.ito4.jp` は旧と同じ公開鍵)。Logto のコネクタの差出人は `logto-domain.php` が書き換える |
| **MFA(生体認証・パスキー)** | **WebAuthn の登録はドメインに結び付く(rpId)ので、ito8795.com で登録した鍵は ito4.jp では使えない。** 切り替えたら、旧ドメイン向けの WebAuthn を外し(Totp は残る)、次のサインインで登録し直してもらう。**MFA を外そうと DB を手で書き換えない** —— 2026-09-17 に sign_in_experiences.mfa の形が崩れて Console の MFA 画面が 500 になり、MFA も効かなくなった(控えの値で戻した) |
| TLS | 外部スキャン(SSL Labs A+ / 国内の診断)の指摘を反映: TLS 1.2 の CBC を外し、鍵交換の群を絞った(§7) |

**全員が一度ログアウトされる。切り替え(§4)は数十秒サイトが途切れる。** 利用の少ない時間に、**新しい APK を配る準備ができてから**(§1)。

**先に済ませること:** [12](12-hardening-2026-09-15.ipynb) **§2〜§4 の配備**(この版のリポジトリで。§0-3 で確かめる)。
**検証機([13](13-local-env.ipynb))で §3-1 と §4-4 の `--apply` を一度通す**(2026-09-17 時点で、本番の Logto に対しては「見るだけ」しか流していない。書き込みの道は未検証)。DNS はレジストリが `dnsv.jp` に切り替わり、`ito4.jp`・`admin.ito4.jp`・`mail.ito4.jp` が `163.43.218.158`(§0-2)。

### このノートに出てくるオプション

**どれも 12 §2 で配備する版にしか無い。** 本番の scripts が古いまま流すと「知らない引数」で止まる(2026-09-17 に実際に踏んだ)ので、
変えるセルはすべて、先頭で版を確かめてから動く(▼ 配備の見張り)。

| どこで | オプション | 意味 |
|---|---|---|
| セルの 1 行目 | `--confirm "文"` | 流す前に `yes` を求める(本番が変わるセルだけ) |
| セルの 1 行目 | `--timeout 秒` | それを過ぎたら止める。既定は 600 秒で、**長くかかる立て直しにだけ**付ける |
| `host-cert.sh issue` | `--domain 名前` | 証明書の名前(置き場も `live/<名前>`) |
| `host-cert.sh issue` | `--also 名前` | 同じ証明書に足す名前(ここでは管理用の `admin.ito4.jp`) |
| `host-cert.sh issue` | `--email アドレス` | 期限切れの警告の宛先 |
| `host-cert.sh issue` | `--dry-run` | 試験用の発行元で手順だけ通す(本物は変えない・発行回数を使わない) |
| `host-domain.sh check/apply` | `--api-resource URL` | audience(`KM_API_RESOURCE`)もその値に移す。**付けなければ今の値のまま** |
| `logto-api-resource.php` | `--from` / `--to` | 移す元と先の API リソースの識別子 |
| `logto-api-resource.php` | `--apply` / `--delete-old` | 付けなければ見るだけ。`--apply` で写す、`--delete-old` で旧いものを消す(写しが揃っていなければ止まる) |
| `logto-domain.php` | `--from` / `--apply` | 旧ドメイン。付けなければ見るだけ、`--apply` で書き換える(新しい名前は `.env` から読む) |
| `docker compose exec` | `-T` / `-u www-data` | 端末を割り当てない(ノートのセルには端末が無い)/ PHP を Apache と同じ利用者で動かす(root だとキャッシュの持ち主が変わる) |
| 行の終わり | `</dev/null` | コンテナのコマンドに標準入力を渡さない(渡すとセルの残りの行を飲み込む) |
| `certbot delete` | `--cert-name` / `--non-interactive` | 消す証明書の名前 / 「本当に消すか」を聞かない(セルでは答えられない) |

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回、次に §0-1 を流す**(`%%host` は `ito4.jp` の名前で繋ぐので、`known_hosts` に要る)。

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |

In [ ]:
# 最初に1回だけ実行する。**本番(ito4.jp)へ繋ぐ**(別のノートで検証機に向けたあとでも、ここで本番に戻す)
import os, sys, pathlib, importlib
os.environ['KM_HOST'] = 'ito4.jp'
# %%host は置き場の持ち主で繋ぐ。2026-09-18 に km → kmops(docs/12 §7-4 B)
os.environ['KM_USER'] = 'kmops'
os.environ['KM_KEY'] = str(pathlib.Path.home() / '.ssh' / 'km_ops')
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
importlib.reload(km_nb)
km_nb.load()

## 0. 前提を確かめる

### 0-1. この PC から ito4.jp の名前で SSH できるようにする

**スクリプトの既定の繋ぎ先は `ito4.jp` になった**(`deploy-to-host.ps1`・`host-setup.ps1`・控えのスクリプト・このノート)。
`known_hosts` に `ito4.jp` が無いと、鍵を確かめる SSH(BatchMode)が止まる。`ssh-keyscan` で取った鍵が、
**`ito8795.com` で既に信頼している鍵と同じときだけ**足す(同じホストなので一致するはず)。

🟡 **手元が変わる** —— この PC にファイルを作る・登録する。本番には触れません。

In [ ]:
%%ps
$kh = Join-Path $env:USERPROFILE '.ssh\known_hosts'
if (& ssh-keygen -F ito4.jp -f $kh 2>$null) { 'known_hosts に ito4.jp は既にあります'; return }
$old = ((& ssh-keygen -lF ito8795.com -f $kh 2>$null) | Select-String 'ED25519' | Select-Object -First 1)
if (-not $old) { throw 'known_hosts に ito8795.com の ED25519 の鍵がありません' }
$oldFp = ("$old" -split ' ') | Where-Object { $_ -like 'SHA256:*' } | Select-Object -First 1
$scan = (& ssh-keyscan -t ed25519 ito4.jp 2>$null) | Where-Object { $_ -and $_ -notmatch '^#' } | Select-Object -First 1
if (-not $scan) { throw 'ito4.jp から鍵を受け取れません(DNS か 22 番)' }
$tmp = New-TemporaryFile
try {
    Set-Content -Path $tmp -Value $scan -Encoding ascii
    $newFp = ((& ssh-keygen -lf $tmp.FullName) -split ' ')[1]
} finally { Remove-Item $tmp -Force }
"ito8795.com で信頼している鍵: $oldFp"
"ito4.jp が出した鍵          : $newFp"
if ($newFp -ne $oldFp) { throw '★ 鍵が違います。足しません(名前が別のホストを指しているか、なりすまし)' }
Add-Content -Path $kh -Value $scan -Encoding ascii
'一致したので known_hosts に ito4.jp を足しました'

### 0-2. DNS(この PC から)

`ito4.jp`・`admin.ito4.jp`・`mail.ito4.jp` が `163.43.218.158`。**CAA** は外部スキャンの指摘(§7)で、無くても動くが足すと Let's Encrypt 以外に証明書を出させない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
nslookup -type=ns ito4.jp a.dns.jp 2>&1 | Select-String 'nameserver'
foreach ($s in '1.1.1.1', '8.8.8.8') {
    foreach ($n in 'ito4.jp', 'admin.ito4.jp', 'mail.ito4.jp') {
        try {
            $ip = (Resolve-DnsName $n -Type A -Server $s -DnsOnly -ErrorAction Stop | Where-Object Type -eq 'A').IPAddress -join ','
            '{0,-8} {1,-14} → {2,-16} {3}' -f $s, $n, $ip, $(if ($ip -eq '163.43.218.158') { 'OK' } else { '★ まだ古い' })
        } catch { '{0,-8} {1,-14} → 引けない({2})' -f $s, $n, $_.Exception.Message }
    }
}
# CAA は Windows の Resolve-DnsName では引けない(型の一覧に CAA が無く、引数の検査で落ちる)。
# 2026-09-18、DNS に足してあるのに「無し」と出ていた。DNS over HTTPS で 2 か所に聞く
foreach ($doh in 'https://cloudflare-dns.com/dns-query', 'https://dns.google/resolve') {
    try {
        $r = Invoke-RestMethod "${doh}?name=ito4.jp&type=CAA" -Headers @{ accept = 'application/dns-json' } -TimeoutSec 10
        $caa = @($r.Answer | Where-Object type -eq 257 | ForEach-Object data)
        '{0,-36} CAA: {1}' -f $doh, $(if ($caa) { ($caa -join ' / ') + $(if ($caa -match 'issue "letsencrypt.org"') { '  OK' } else { '  ★ letsencrypt.org が無い' }) } else { '無し(§7 の推奨: 0 issue "letsencrypt.org")' })
    } catch { '{0,-36} CAA: 聞けない({1})' -f $doh, $_.Exception.Message }
}


### 0-3. ホストの状態

**★ が 1 つでも出たら、このセルは失敗で終わる。先へ進まない。** 「配備が古い」は [12](12-hardening-2026-09-15.ipynb) §2 の配備(この版のリポジトリ)が済んでいないということ ——
そのまま進むと §2 以降のセルがすべて「知らない引数」で止まる(下の各セルの見張りも同じ理由で止める)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
bad=0
echo "ホスト: $(hostname -I | awk '{print $1}')"
grep -E '^(KM_DOMAIN|KM_ADMIN_DOMAIN|KM_API_RESOURCE|COMPOSE_FILE)=' .env
if grep -q -- '--api-resource' scripts/host-domain.sh && grep -q -- '--also' scripts/host-cert.sh && [ -f src/scripts/logto-api-resource.php ] && grep -q '^ssl_ciphers ECDHE' nginx/default.conf.template; then
  echo "配備: このノートの版があります"
else
  echo "★ 配備が古い(host-domain.sh に --api-resource が無い・logto-api-resource.php が無い など)。12 §2 の配備をこの版で"; bad=1
fi
if docker network ls --format '{{.Name}}' | grep -q 'edge$'; then echo "12 §2(網の分割): 済"; else echo "★ 12 §2(網の分割・read_only)がまだです"; bad=1; fi
# §4-1 より前に KM_ADMIN_DOMAIN があると、12 §2 の立て直しの時点で管理画面が admin.ito4.jp へ移り、証明書も Logto の登録も無いので入れなくなる
if grep -q '^KM_ADMIN_DOMAIN=' .env && ! grep -q '^KM_DOMAIN=ito4\.jp$' .env; then
  echo "★ .env に KM_ADMIN_DOMAIN があります(§4-1 より前)。12 §2 の立て直しの前に消す: sed -i '/^KM_ADMIN_DOMAIN=/d' .env"; bad=1
fi
for n in ito4.jp admin.ito4.jp mail.ito4.jp; do
  echo "$n → $(getent ahostsv4 "$n" | awk '{print $1}' | sort -u | tr '\n' ' ')"
done
echo "DKIM の鍵: $(docker exec km-mailserver ls /etc/opendkim/keys </dev/null | tr '\n' ' ')"
[ "$bad" = 0 ] || { echo; echo "★ があるので先へ進まないでください"; exit 1; }

## 1. Android の新しい APK を作っておく(切り替えの前に)

**古い APK は切り替えた時点で繋がらなくなる**(旧ドメインも旧 audience も無くなる)。先に作って、切り替えたらすぐ配れるようにする。
手元の `Test` は既定が `ito4.jp`・`https://ito4.jp/api` になっている。visitor と admin の**両方**:

```
.\gradlew.bat testVisitorDebugUnitTest testAdminDebugUnitTest assembleVisitorRelease assembleAdminRelease
```

単体試験は **debug にしか無い**(`testVisitorReleaseUnitTest` は「そんなタスクは無い」で止まる。2026-09-18)。試験は debug で流し、APK は release で作る。

切り替えたあと、管理画面の「ダウンロード」の APK を差し替える(§4-6)。

## 2. 証明書を取る(ito4.jp と admin.ito4.jp)

**まだ使わない**(nginx は §4 の切り替えまで `live/ito8795.com` を使う)。**2 つのセルの `MAIL=you@example.com` を、期限切れの警告を受け取る自分のアドレスに書き換えてから流す**
(いまのアカウントには連絡先が保存されていないので読み出せない)。まず `--dry-run`(試験用の発行元。発行回数を消費しない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "ito4.jp の証明書(admin.ito4.jp 入り)を、試験用の発行元で試します(本物は変えません)"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
MAIL=you@example.com   # ← 期限切れの警告を受け取るアドレス(Let's Encrypt に登録される)
case "$MAIL" in *@example.jp|'') echo "★ MAIL= を自分のアドレスに書き換えてから流してください"; exit 1 ;; esac
# --also: 同じ証明書に管理用の名前も入れる / --dry-run: 試験用の発行元(本物は変えない)
./scripts/host-cert.sh issue --domain ito4.jp --also admin.ito4.jp --email "$MAIL" --dry-run </dev/null

試験が通ったら本物を取る。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "ito4.jp の証明書(admin.ito4.jp 入り)を発行します(nginx はまだ使いません)"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
MAIL=you@example.com   # ← 期限切れの警告を受け取るアドレス(Let's Encrypt に登録される)
case "$MAIL" in *@example.jp|'') echo "★ MAIL= を自分のアドレスに書き換えてから流してください"; exit 1 ;; esac
# --also: 同じ証明書に管理用の名前も入れる(取れた名前と期限は、発行の出力の最後に出る)
./scripts/host-cert.sh issue --domain ito4.jp --also admin.ito4.jp --email "$MAIL" </dev/null

## 3. Logto を準備する(切り替えの前に。**誰も使っていないものを足すだけ**)

### 3-1. 新しい API リソース(audience)を作り、スコープとロールを写す

`https://ito4.jp/api` のリソースを作り、`https://ito8795.com/api` のスコープ(`admin:users:read` など 5 つ)を同じ名前で作り、
それを持つロール(`kosenmap-admin`・`kosenmap-staff`)に新しい方も割り当てる。**旧いリソースは残す**(§6 で消す)——
残っている間は、どちらの audience でもトークンを出せるので、切り替えの前後で誰も締め出されない。まず見るだけ。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# オプションなし(--apply が無い)= 見るだけ
docker compose exec -T -u www-data web php scripts/logto-api-resource.php --from=https://ito8795.com/api --to=https://ito4.jp/api </dev/null

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto に https://ito4.jp/api のリソースを作り、スコープとロールの割り当てを写します(旧いリソースは残す)"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# --apply: 新しいリソースを作り、スコープとロールの割り当てを写す(旧いリソースは消さない)
docker compose exec -T -u www-data web php scripts/logto-api-resource.php --from=https://ito8795.com/api --to=https://ito4.jp/api --apply </dev/null

### 3-2. 管理用の戻り先を足す(Console の画面で)

Console(**いまは** `https://ito8795.com:3002`)→ アプリケーション → **管理画面の Web アプリ**(`.env` の `LOGTO_APP_ID` のもの):

| 欄 | 足す値(`ito8795.com` の分は §4-4 の `logto-domain.php` が `ito4.jp` へ書き換える) |
|---|---|
| リダイレクト URI | `https://admin.ito4.jp/callback.php` |
| サインアウト後のリダイレクト URI | `https://admin.ito4.jp/` |
| CORS の許可オリジン(欄があれば) | `https://admin.ito4.jp` |

**保存を押す。** 足していないと、管理画面のサインインが `redirect_uri` の不一致で落ちる。

> **切り替えの前に必ず。** 切り替えたあとは Console(dmin.ito4.jp:3002)が管理画面のサインインの内側に入るので、**ここを忘れると Console から足しに行けない**(2026-09-17 に実際に抜けて、invalid_redirect_uri で管理画面に入れなくなった。Management API から足して直した)。

## 4. 切り替える

### 4-1. 管理用の名前を書き、切り替えられるか調べる

`.env` に `KM_ADMIN_DOMAIN=admin.ito4.jp` を書き(控えを取る。**コンテナはまだ作り直さない**)、`host-domain.sh check ito4.jp --api-resource https://ito4.jp/api` で
`.env` の書き換え方・compose が読む値・証明書の 2 つの名前・DNS・DKIM の鍵を見る。**★ が出たら 4-2 へ進まない。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm ".env に KM_ADMIN_DOMAIN=admin.ito4.jp を書き(控えを取る。まだ立て直さない)、ito4.jp へ移せるか調べます"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
set -eu
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)-before-admin"
sed -i '/^KM_ADMIN_DOMAIN=/d' .env
printf 'KM_ADMIN_DOMAIN=%s\n' admin.ito4.jp >> .env
# check: 調べるだけ / --api-resource: audience もこの値に移す前提で調べる
./scripts/host-domain.sh check ito4.jp --api-resource https://ito4.jp/api </dev/null

### 4-2. `.env` を書き換え、DKIM の鍵を複製する

`KM_DOMAIN=ito4.jp`・`KM_API_RESOURCE=https://ito4.jp/api` を書き、旧ドメインの URL の行をコメントにする。DKIM の `ito8795.com.private` を `ito4.jp.private` として複製する。
**まだ立て直さない。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm ".env を ito4.jp(audience も)に書き換え、DKIM の鍵を複製します(まだ立て直しません)"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
grep -q '^KM_ADMIN_DOMAIN=admin\.ito4\.jp$' .env || { echo "★ 先に 4-1 を流してください(KM_ADMIN_DOMAIN がありません)"; exit 1; }
# apply: .env を控えてから書き換える(立て直しはしない) / --api-resource: KM_API_RESOURCE もこの値にする
./scripts/host-domain.sh apply ito4.jp --api-resource https://ito4.jp/api </dev/null

### 4-3. 立て直す

**ここから旧ドメインと古い APK は使えなくなる。** 数十秒途切れ、全員が一度ログアウトされる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "ito4.jp で立て直します(数十秒途切れる。旧ドメインと古い APK はここから使えない)" --timeout 900
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# 4-2 が済んでいなければ立て直さない(古い .env のまま立て直しても何も移らない)
grep -q '^KM_DOMAIN=ito4\.jp$' .env || { echo "★ .env の KM_DOMAIN が ito4.jp ではありません。先に 4-2 を流してください"; exit 1; }
docker compose up -d </dev/null
sleep 20
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null

### 4-4. Logto の戻り先とメールの差出人を ito4.jp へ

`logto-domain.php` が、ホスト名が `ito8795.com` の登録を `ito4.jp` に書き換える:
Web アプリの戻り先・webhook(アカウント削除)・サインイン画面の規約などの URL・**メールのコネクタの差出人とテンプレートの URL**。
**差出人が旧のままだと mailserver が拒み、確認コードが届かない** —— 4-3 のすぐあとに流す。まず一覧だけ。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# オプションなし(--apply が無い)= 見るだけ。新しい名前は .env(APP_URL)から読むので、4-3 の立て直しの後に流す
docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito8795.com </dev/null

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto に登録された ito8795.com の URL とメールの差出人を ito4.jp へ書き換えます"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# --apply: 書き換える(読み直して、旧ドメインが残っていないかも見る)
docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito8795.com --apply </dev/null

### 4-5. 確かめる

| 見るもの | 期待 |
|---|---|
| `https://ito4.jp/` | 200 |
| `https://ito4.jp/admin/` | 308 → `https://admin.ito4.jp/admin/` |
| `https://admin.ito4.jp/admin/login.php` | 200 |
| `https://admin.ito4.jp/` | 308 → `https://ito4.jp/` |
| `https://ito4.jp:3001/…/openid-configuration` | 200、issuer は `https://ito4.jp:3001/oidc` |
| web の audience | `https://ito4.jp/api` |
| TLS 1.2 の CBC(`ECDHE-ECDSA-AES128-SHA`) | **繋がらない** |
| TLS 1.2 の GCM(`ECDHE-ECDSA-AES128-GCM-SHA256`) | 繋がる |
| 証明書 | `ito4.jp`・`admin.ito4.jp`、nginx が出しているものと一致(`host-cert.sh status`) |
| mailserver | 差出人 `ito4.jp`、鍵 `ito4.jp.private` |

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
for u in https://ito4.jp/ https://ito4.jp/admin/ https://admin.ito4.jp/admin/login.php https://admin.ito4.jp/ \
         https://ito4.jp:3001/oidc/.well-known/openid-configuration; do
  # curl: -s 経過を出さない / -o /dev/null 本文を捨てる / -m 15 秒で諦める / -w 応答の番号と転送先だけ出す
  printf '%-62s → ' "$u"; curl -s -o /dev/null -m 15 -w '%{http_code} %{redirect_url}\n' "$u"
done
echo "issuer  : $(curl -s -m 15 https://ito4.jp:3001/oidc/.well-known/openid-configuration | grep -o '"issuer":"[^"]*"')"
echo "audience: $(docker compose exec -T web printenv KOSENMAP_LOGTO_AUDIENCE </dev/null)"
# TLS 1.2 で、その暗号だけを名指しして繋いでみる(繋がれば、その暗号を受けている)
tls() { printf '' | timeout 10 openssl s_client -connect ito4.jp:443 -servername ito4.jp -tls1_2 -cipher "$1" 2>/dev/null | grep -q '^ *Cipher *: *'"$1"'$' && echo "繋がる" || echo "繋がらない"; }
echo "TLS 1.2 CBC (ECDHE-ECDSA-AES128-SHA)        : $(tls ECDHE-ECDSA-AES128-SHA)(繋がらないが正しい)"
echo "TLS 1.2 GCM (ECDHE-ECDSA-AES128-GCM-SHA256) : $(tls ECDHE-ECDSA-AES128-GCM-SHA256)(繋がるが正しい)"
echo
./scripts/host-cert.sh status </dev/null
echo
echo "mailserver の差出人: $(docker exec km-mailserver sh -c 'echo "$ALLOWED_SENDER_DOMAINS"' </dev/null) / 鍵: $(docker exec km-mailserver ls /etc/opendkim/keys </dev/null | tr '\n' ' ')"
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null

### 4-6. 画面と端末で確かめる

1. **ブラウザ:** `https://admin.ito4.jp/admin/` にサインインし、戻ってこられる(確認コードのメールが届く)。ユーザー一覧が開ける(`admin:users:read`)
2. **メール:** Console(`https://admin.ito4.jp:3002`)→ コネクタ → メール(SMTP)→ **テスト送信**を自分の Gmail へ。「メッセージのソースを表示」で **SPF・DKIM・DMARC がすべて PASS**、差出人が `noreply@ito4.jp`
3. **Android:** §1 の新しい APK(visitor・admin)でサインイン → 地図の取得 → admin 版でスタッフの機能(`staff:event:access`)
4. **APK の差し替え:** 管理画面の「ダウンロード」に新しい APK を置く

## 5. reCAPTCHA とこの PC(画面と手元)

**reCAPTCHA:** Google reCAPTCHA の管理画面 → このサイトのキー → ドメインを **`ito4.jp` と `admin.ito4.jp`** にし、`ito8795.com` を外す。
`https://ito4.jp/contact.php` から 1 通送って確かめる(足していないと「reCAPTCHA の検証に失敗」で落ちる)。

**週次の控えのタスクを登録し直す。** いまのタスクは `-HostName ito8795.com` を引数に持っている(既定を変えても効かない)。
既定が `ito4.jp` になった `register-backup-task.ps1` で上書きする([03-backup](03-backup.ipynb) の登録と同じ)。

🟡 **手元が変わる** —— この PC にファイルを作る・登録する。本番には触れません。

In [ ]:
%%ps
.\register-backup-task.ps1
Get-ScheduledTask | Where-Object TaskName -like 'KosenMap*' | ForEach-Object { "$($_.TaskName): $($_.Actions.Arguments)" }

## 6. 後片付け(4-6 まで確かめてから)

### 6-1. 旧い API リソースを消す

新しいリソースへの写しが揃っていなければ止まる。消すと旧いスコープとロールへの割り当ても一緒に消え、`https://ito8795.com/api` のトークンは出なくなる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の旧い API リソース https://ito8795.com/api を消します(写しが揃っていなければ止まる)"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
# --delete-old: 新しいリソースへの写しが揃っているときだけ、旧いリソースを消す
docker compose exec -T -u www-data web php scripts/logto-api-resource.php --from=https://ito8795.com/api --to=https://ito4.jp/api --delete-old </dev/null

### 6-2. 旧い証明書を消す

`live/ito8795.com` はもう使っていない。残すと certbot が更新を試み続け、旧ドメインの DNS を外したあと毎回失敗する。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "使わなくなった ito8795.com の証明書を certbot から消します"
# ▼ 配備の見張り: 本番の scripts がこのノートの版でなければ、何も変えずに止める(12 §2 で配備する)
if ! grep -q -- '--api-resource' scripts/host-domain.sh || ! grep -q -- '--also' scripts/host-cert.sh || [ ! -f src/scripts/logto-api-resource.php ]; then
  echo "★ 本番の scripts が古い版です。先に 12 §2 で配備してください(このセルは何も変えていません)"; exit 1
fi
grep -q '^KM_DOMAIN=ito4\.jp$' .env || { echo "★ まだ切り替えていません(KM_DOMAIN が ito4.jp ではない)。消しません"; exit 1; }
./scripts/host-cert.sh status </dev/null | grep -E '^(★ |注 )?ito4\.jp' || { echo "★ ito4.jp の証明書が見えません。消しません"; exit 1; }
# --cert-name: 消す証明書の名前 / --non-interactive: 「本当に消すか」を聞かない(セルでは答えられない)
docker compose exec -T certbot certbot delete --cert-name ito8795.com --non-interactive </dev/null
# 見るだけ: Logto に ito8795.com が残っていないか
docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito8795.com </dev/null

### 6-3. 残すもの・人が決めるもの

- **mailserver の `ito8795.com.private`**: 残しておいて害は無い(差出人の許可に無いので使われない)
- **`ito8795.com` のドメインと DNS**: 手放すかどうかは人が決める。**A レコードを残したままにすると**、`https://ito8795.com` は証明書の名前が合わずにエラーになる(HSTS を覚えたブラウザは先へ進めない)。案内を出すなら、旧ドメインだけを 301 で転送する設定と証明書を足す必要がある
- **SPF・DMARC・DKIM の旧ドメインの記録**: 送らなくなったので、ドメインを手放すまで残しても害は無い

## 7. 外部スキャンの指摘(2026-09-17、`ito8795.com` で実施)と対応

対象は同じ nginx なので、`ito4.jp` にもそのまま当てはまる。

| 出どころ | 指摘 | 判断 | 対応 |
|---|---|---|---|
| SSL Labs | 総合 **A+**(TLS 1.3・耐量子の鍵交換 X25519MLKEM768・HSTS 1 年・チェーン問題なし・既知の脆弱性なし) | — | — |
| SSL Labs | TLS 1.2 の **CBC**(`ECDHE-ECDSA-AES128/256-CBC-SHA/SHA256/SHA384`・`CAMELLIA-CBC`)が **WEAK** | 直す。使う端末は無い(アプリは minSdk 29 で TLS 1.3) | `ssl_ciphers` を ECDHE の GCM・ChaCha20 だけに(`nginx/default.conf.template`)。4-5 で確かめる |
| SSL Labs | 鍵交換の群に `ffdhe2048/3072`・`secp521r1`・`x448` | 直す(使われない) | `ssl_ecdh_curve X25519MLKEM768:X25519:prime256v1:secp384r1` |
| SSL Labs | **DNS CAA: No** | 足すとよい(他の認証局に証明書を出させない) | **DNS の画面で** `ito4.jp. CAA 0 issue "letsencrypt.org"` を足す(0-2 で確かめる) |
| SSL Labs | OCSP stapling: No | 対応しない | Let's Encrypt は 2025 年に OCSP を終え、失効情報は CRL だけになった。stapling する相手が無い |
| SSL Labs | HSTS preload に未登録 | 今はしない | 取り消しが難しい。`www.ito4.jp` が証明書に無いので includeSubDomains も付けられない(`nginx/km/hsts-enable.conf.example`) |
| 国内の診断 | 鍵長 256 bit(「2048 bit 以上を推奨」) | **誤検知** | ECDSA P-256 は RSA 3072 bit 相当(SSL Labs も同じ評価)。RSA の基準を当てはめている |
| 国内の診断 | 残り期間が少ない(2 か月 9 日) | 問題なし | Let's Encrypt は 90 日で、certbot が残り 30 日で自動更新する。残り 14 日を切ると certbot の healthcheck が unhealthy になる |
| 国内の診断 | TLS 1.0・1.1・SSLv2/3 は不可、脆弱な暗号(RC4・DES・DSS・EXPORT)は拒否 | — | — |

直したあと、SSL Labs を `ito4.jp` でもう一度流す(「Clear cache」)。TLS 1.2 の一覧に WEAK が無いことを見る。

## 9. 2026-09-17 の結果と残り

**切り替えは済んだ**(利用者が流した)。そのあと起きたことと、残っている作業。**残りの一覧と番号は [12](12-hardening-2026-09-15.ipynb) §0「利用者待ちの作業」にまとめてある。**

| 起きたこと | 原因 | 直したこと |
|---|---|---|
| `ito4.jp` を開くと `ito8795.com` へ飛ぶ | 切り替えの前に受け取った 308(http → 旧ドメイン)をブラウザが覚えていた | ブラウザのキャッシュを消した(サーバーは正しかった) |
| 管理画面のサインインが `invalid_redirect_uri` | §3-2(`admin.ito4.jp` の戻り先)が抜けていた。切り替えたあとは Console に入れず足せない | Management API で足した。§3-2 に注意を書いた |
| 生体認証で入れない → MFA の設定を DB で手直し → `/console/mfa` が 500 | WebAuthn はドメインに結び付く。`sign_in_experiences.mfa` が `mode`(1.43 は `policy`)になった | 控えの値に戻し、旧ドメインの WebAuthn を外した。冒頭の表に注意を書いた |
| ログアウトを押しても画面がそのまま | CSP の `form-action 'self'` が Logto(:3001)への 302 を止めた | サインアウトはページを返して meta refresh で移る |
| 一般アカウントで管理画面が「認証の設定が未完了」 | SDK がトークンを base64(base64url ではない)で読み、`-`・`_` を含むトークンで落ちた | 子クラス `KmLogtoClient` で読み方を差し替えた |

**このノートの残り(2026-09-18 に実測し直した):** **Android のリリース版の配布(§1・#2)と、`ito8795.com` の解約(#15)だけ。**
旧い証明書の削除(§6-2・#4)・テスト送信(§4-6・#8)・reCAPTCHA(§5・#9)・SSL Labs(#10)・CAA(#11)は済み。

## 8. 戻し方

**§6 の前なら戻せる。** §6-1 で旧い API リソースを消したあとは、旧い audience へは戻せない(作り直しになる)。

| 段 | 戻し方 |
|---|---|
| §2 証明書 | 何もしなくてよい(使っていない)。消すなら `certbot delete --cert-name ito4.jp` |
| §3-1 新しいリソース | 残しておいて害は無い |
| §3-2 Logto に足した URL | 残しておいて害は無い |
| §4-1 `KM_ADMIN_DOMAIN` | `.env.bak-<日時>-before-admin` を `cat … > .env` で戻す |
| §4-2 `.env` | `host-domain.sh apply` が出した控え: `cat .env.bak-<日時> > .env && docker compose up -d`(DKIM の複製は残して害は無い) |
| §4-4 Logto | `logto-domain.php --from=ito4.jp --to=ito8795.com --apply`(`admin.ito4.jp` の分は Console で消す) |
| §1 Android | 古い APK は、上をすべて戻せばまた動く |